# ClaimCheck on Google Colab
Run the cells **from top to bottom, one at a time**. Wait for each cell to finish (a green tick) before the next.
Before you start: (1) upload `claimcheck.zip` to the top level of your Google Drive ("My Drive"),
(2) click the key icon on the left (Secrets) and add: `LLM_BASE_URL`, `LLM_API_KEY`, `LLM_MODEL`, `EMBED_MODEL`
(optional: `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`) and switch **Notebook access ON** for each.
Use Runtime > Change runtime type > T4 GPU (free) only if you want the reranker to run faster; CPU also works.

## Step 1. Set up the project (about 3 minutes)

In [ ]:
from google.colab import drive, userdata
import os, shutil
drive.mount('/content/drive')
if not os.path.exists('/content/claimcheck'):
    shutil.unpack_archive('/content/drive/MyDrive/claimcheck.zip', '/content')
%cd /content/claimcheck
!pip install -q -r requirements.txt -r requirements-rerank.txt
# keep results on Drive so a Colab restart does not delete them
os.environ['CC_DATA_DIR'] = '/content/drive/MyDrive/claimcheck_data'
for name in ['LLM_BASE_URL','LLM_API_KEY','LLM_MODEL','EMBED_MODEL','EMBED_PROVIDER','LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY']:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        print('not set:', name)
os.environ.setdefault('LANGFUSE_HOST', 'https://cloud.langfuse.com')
# our endpoint has no /embeddings route, so embeddings run here (mentor-approved fallback)
os.environ.setdefault('EMBED_PROVIDER', 'local')

## Step 2. Check the endpoint (do this on day 1)
Expected: a chat reply, 'JSON mode supported: True/False', and an embedding size. If it fails, tell your mentor immediately.
Embeddings run inside Colab with `BAAI/bge-small-en-v1.5` (`EMBED_PROVIDER=local`), so expect 'embeddings provider: local' and size 384.

In [ ]:
!python scripts/smoke_test.py

## Step 3. Build the data (600 documents, 15 members, 98 evaluation questions) and run the 57 tests

In [ ]:
!python scripts/build_data.py
!python -m pytest tests -q

## Step 4. WEEK 1: prompting strategies (compares 4 strategies on 30 fact and 8 cost questions)
Save the table. Read `week1_failures.csv` and write down how the model breaks.

In [ ]:
!python scripts/run_week1.py --n 30
import pandas as pd
pd.read_csv(os.environ['CC_DATA_DIR'] + '/results/week1_strategies.csv')

## Step 5. WEEK 2: retrieval experiment (chunking x BM25 / dense / hybrid / reranker)
The first run embeds the documents (a few minutes). The result table is your recall@k evidence.

In [ ]:
!python scripts/run_retrieval_exp.py
pd.read_csv(os.environ['CC_DATA_DIR'] + '/results/retrieval_experiment.csv')

Pick the best chunking from the table, then build that index. Replace `sections` if another one won, and set `CHUNKING` to the same name.

In [ ]:
os.environ['CHUNKING'] = 'sections'
!python scripts/build_index.py --chunking sections

## Step 6. Try the agent on single questions

In [ ]:
!python scripts/ask.py "What is the ICU cap per day on the Gold plan in zone C?"
!python scripts/ask.py "I have a ₹1,20,000 hospital bill with 3 days in a ₹9,000 per day room. How much will I pay?" --user M003

## Step 7. WEEK 3: evaluation
First a small test of 10 items. Then the full set three times (run 1, 2, 3), then the single-prompt ablation.

In [ ]:
!python scripts/run_eval.py --name agent --run 1 --limit 10

In [ ]:
os.environ['USE_CACHE'] = '0'   # official runs must really call the model
for run in (1, 2, 3):
    !python scripts/run_eval.py --name agent --run {run}
!python scripts/summarize_runs.py agent

In [ ]:
for run in (1, 2, 3):
    !python scripts/run_eval.py --name single_rag --mode single --run {run}
!python scripts/summarize_runs.py single_rag

## Step 8. WEEK 4: security tests (writes `security_report.md` for your report)

In [ ]:
!python scripts/run_security.py

## Step 9. Launch the web app and get a public link for the demo
The link stops working when this cell stops or Colab disconnects.

In [ ]:
os.environ['USE_CACHE'] = '1'   # the app may use the answer cache again
import subprocess, time, re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
app = subprocess.Popen(['streamlit','run','app/streamlit_app.py','--server.port=8501','--server.headless=true'])
time.sleep(10)
tunnel = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8501'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        print('OPEN THIS LINK:', m.group(0)); break